# 05 — Model: forecasting TCAS70 program min scores

**Consumes** the 48-code balanced Chula panel from `nb/01_clean`, plus the
cohort difficulty index from `nb/03`/`src/normalize.py`.

**Emits** `data/processed/forecast_tcas70.csv` — 48 programs, point estimate
and 80%/95% intervals.

**Concludes** the partially-pooled hierarchical model **loses** to trivial
baselines, and last-value-carried-forward ships. The more freedom a method has
to fit a per-program trend, the worse it forecasts. This is reported as the
result, not worked around.

---

### Read this before the numbers

There are **four observations per program**. Leave-last-year-out leaves three
points to fit an intercept and a slope — one residual degree of freedom. The
per-program fit is nearly saturated, so partial pooling is not a refinement
here; it is the only thing that could make per-program slopes legitimate.

Two consequences stated up front, so nothing below looks like a surprise:

- **The interval will be wide, and that is the correct answer.** It is not
  tuned until it narrows.
- **If the model loses to last-value-carried-forward, LVCF ships.** At n = 4
  that is a plausible outcome and reporting it honestly beats a model that
  wins by construction.

In [1]:
import sys; sys.path.insert(0, "..")
import pandas as pd
import src.model as M

panel = M.build_panel()
print(f"panel: {len(panel)} rows = {panel.code.nunique()} programs x {panel.year.nunique()} years")
panel[["code", "year", "faculty", "min_score", "applied", "passed", "diff_alevel"]].head(4)

panel: 192 rows = 48 programs x 4 years


,code,year,faculty,min_score,applied,passed,diff_alevel
0,10010121300001A,66,คณะวิศวกรรมศาสตร์,54.1500,2424.0,360.0,35.650245
1,10010121300001A,67,คณะวิศวกรรมศาสตร์,53.9610,1119.0,380.0,34.291793
2,10010121300001A,68,คณะวิศวกรรมศาสตร์,59.9831,1417.0,390.0,36.976195
3,10010121300001A,69,คณะวิศวกรรมศาสตร์,54.6525,857.0,305.0,36.362859


## 1. Two design decisions that constrain everything

**Seats are absent.** `รับ` cannot be summed and its best reconstruction falls
below actual admits in all four years (Trap 1). No feature derived from it
enters the model.

**Covariates must be lagged.** A genuine TCAS70 forecast cannot use TCAS70's
applications or admits — they do not exist yet. Using contemporaneous
covariates would make this a back-fit rather than a forecast. Lagging them
costs a year of panel depth (4 points becomes 3), which at this sample size is
expensive enough that the primary model is a pure trend model.

## 2. Baselines first

Three, per the spec:

- **`lvcf`** — last value carried forward. The one to beat.
- **`program_mean`** — the program's own average over the training years.
- **`program_ols`** — per-program least squares on year, *no pooling*. Included
  to show what unpooled trend estimation does at this sample size.

And two model variants:

- **`hierarchical`** — random intercept **and slope** per program, shrunk
  toward the population, faculty as a fixed effect. The spec's primary model.
- **`hier_no_slope`** — pools the level only, single global slope. The
  controlled comparison that isolates whether per-program trends earn their keep.

`statsmodels` `MixedLM` is used because neither `numpyro` nor `pymc` is
installed here — the spec's sanctioned fallback. It is a two-level
approximation to the spec's three-level design; statsmodels cannot cleanly
express random slopes nested at two levels.

## 3. Leave-last-year-out — fit on 66–68, predict 69

In [2]:
summary, per_faculty, errors = M.backtest(panel)
summary

,MAE,RMSE,bias,n
program_mean,4.892,6.664,3.205,48.0
hier_no_slope,5.089,6.631,3.340,48.0
lvcf,5.388,6.607,3.450,48.0
hierarchical,6.660,8.043,3.340,48.0
program_ols,8.021,9.726,3.340,48.0


**The hierarchical model loses.** It is 36% worse than the best baseline on
MAE. And the ordering is not random — it is monotone in how much freedom each
method has to fit a per-program trend:

| method | per-program trend | MAE |
|---|---|---|
| `program_ols` | free slope, no pooling | 8.02 |
| `hierarchical` | slope, shrunk | 6.66 |
| `hier_no_slope` | level only, global slope | 5.09 |
| `program_mean` | none | 4.89 |

Partial pooling does exactly what it is supposed to — the shrunk-slope model
beats the unpooled one by 1.4 MAE. It just is not enough. **At four points per
program, estimating a trend costs more than it earns.**

In [3]:
# Every method over-predicts the held-out year by ~3 points.
summary[["MAE", "RMSE", "bias"]].round(2)

,MAE,RMSE,bias
program_mean,4.89,6.66,3.20
hier_no_slope,5.09,6.63,3.34
lvcf,5.39,6.61,3.45
hierarchical,6.66,8.04,3.34
program_ols,8.02,9.73,3.34


That shared positive bias is a clue: TCAS69 *fell*, and no method saw it
coming. Which raises the question of whether a single held-out year is enough
to choose on at all.

## 4. Per-faculty error — a good average can hide a faculty the model cannot touch

In [4]:
per_faculty.sort_values("lvcf", ascending=False)

,lvcf,program_mean,program_ols,hierarchical,hier_no_slope,n
faculty,,,,,,
คณะพาณิชยศาสตร์และการบัญชี,12.68,8.14,15.06,13.83,8.28,3
คณะนิเทศศาสตร์,8.19,5.30,11.97,10.78,5.43,1
คณะครุศาสตร์,8.07,5.08,11.02,9.97,5.22,1
คณะจิตวิทยา,5.91,4.02,6.91,6.41,4.16,1
คณะวิศวกรรมศาสตร์,5.44,2.03,8.05,6.91,2.49,10
คณะวิทยาศาสตร์,5.34,7.17,8.26,6.53,7.42,17
คณะสหเวชศาสตร์,4.69,1.41,9.89,8.15,1.26,4
คณะวิทยาศาสตร์การกีฬา,4.49,12.63,9.99,5.89,12.50,1
สำนักวิชาทรัพยากรการเกษตร,4.25,15.67,15.65,9.97,15.54,1


Errors are wildly uneven. พาณิชยศาสตร์และการบัญชี is off by 8–15 points under
every method; รัฐศาสตร์ and สถาปัตยกรรมศาสตร์ by 1–3. Note also that most
faculties contribute a single program to the panel — วิทยาศาสตร์ (17) and
วิศวกรรมศาสตร์ (10) carry more than half of it, so per-faculty figures for
n = 1 faculties are single observations, not averages.

## 5. One held-out year is not enough to choose on

The spec asks for leave-last-year-out, which is a single 68→69 transition. But
the panel median oscillates rather than trends:

In [5]:
panel.groupby("year")["min_score"].median().round(2).to_frame("panel median min")

,panel median min
year,
66,58.71
67,53.94
68,57.97
69,53.50


In [6]:
long, pooled = M.rolling_backtest(panel)
long.pivot(index="method", columns="origin", values="bias").round(2)

origin,67→68,68→69
method,,
hier_no_slope,-1.07,3.34
hierarchical,-1.07,3.34
lvcf,-0.60,3.45
program_mean,-0.37,3.20
program_ols,-1.07,3.34


**Every method's bias flips sign between the two origins** — negative into the
rising year 68, positive into the falling year 69. So a single holdout bakes
that one year's direction into both the chosen winner and the interval width.

Rolling the origin gives a second transition. It is still only two, and the
67→68 origin trains on just two years, which penalises the trend methods
further — both caveats worth carrying.

In [7]:
long.pivot(index="method", columns="origin", values="MAE").round(2)

origin,67→68,68→69
method,,
hier_no_slope,7.46,5.09
hierarchical,6.81,6.66
lvcf,5.89,5.39
program_mean,7.13,4.89
program_ols,7.24,8.02


In [8]:
pooled

,MAE,RMSE,n
method,,,
lvcf,5.638,7.132,96.0
program_mean,6.012,8.069,96.0
hier_no_slope,6.277,8.118,96.0
hierarchical,6.734,8.203,96.0
program_ols,7.630,9.267,96.0


This changes the recommendation. On the single spec holdout, `program_mean`
wins (4.89). Across both origins, **`lvcf` wins (5.638)** — and it is the only
method that is stable at both (5.89 and 5.39, versus program_mean's 7.13 and
4.89).

So `program_mean`'s apparent win owes itself to which way one particular year
moved. Checking whether it was ever a real win:

In [9]:
pd.Series(M.compare_top_two(panel, "lvcf", "program_mean")).round(3)

MAE lvcf                   5.388
MAE program_mean           4.892
difference                 0.496
paired t p                 0.381
wilcoxon p                 0.137
boot lo                   -0.602
boot hi                    1.546
program_mean better on    31.000
of                        48.000
dtype: float64

It was not. The gap is **not statistically distinguishable** — paired t
p = 0.38, Wilcoxon p = 0.14, and the bootstrap CI for the difference spans
zero. Two methods tied on a single year, separated on two years by stability.

**Shipping `lvcf`**, which is also the spec's stated fallback.

## 6. Forecast

Intervals are **empirical** — quantiles of the shipped method's pooled
one-year-ahead error distribution across both origins, not the model's own
variance estimate. With four points per program the model's internal
uncertainty is not credible; the rolling-origin errors are a demonstrated
out-of-sample record.

In [10]:
result = M.run(verbose=False)
forecast = result["forecast"]
print(f"shipping: {result['winner']}  ·  beat_baseline={result['beat']}")
print(f"mean 80% width: {(forecast.hi80 - forecast.lo80).mean():.1f} points")
print(f"mean 95% width: {(forecast.hi95 - forecast.lo95).mean():.1f} points")

shipping: lvcf  ·  beat_baseline=False
mean 80% width: 16.8 points
mean 95% width: 27.3 points


In [11]:
forecast[["program", "faculty", "point", "lo80", "hi80", "lo95", "hi95"]].head(8).round(1)

,program,faculty,point,lo80,hi80,lo95,hi95
0,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมศา...,คณะวิศวกรรมศาสตร์,54.7,43.9,60.7,35.4,62.6
1,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมคอ...,คณะวิศวกรรมศาสตร์,65.9,55.2,72.0,46.7,73.9
2,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมคอ...,คณะวิศวกรรมศาสตร์,61.1,50.3,67.1,41.8,69.1
3,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมนิ...,คณะวิศวกรรมศาสตร์,51.9,41.1,57.9,32.6,59.8
4,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมโยธา,คณะวิศวกรรมศาสตร์,54.9,44.1,60.9,35.6,62.9
5,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมโล...,คณะวิศวกรรมศาสตร์,49.7,38.9,55.7,30.4,57.7
6,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมสำรวจ,คณะวิศวกรรมศาสตร์,52.0,41.2,58.0,32.7,59.9
7,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมสิ...,คณะวิศวกรรมศาสตร์,52.1,41.3,58.1,32.8,60.0


An 80% interval **16.8 points wide** on a 0–100 scale is the honest answer at
n = 4, and it is still optimistic — it is calibrated on two year transitions,
both of which are in the training data's own regime.

The intervals are asymmetric because the pooled errors are not centred: the
shipped method over-predicts on average (+1.4 across both origins), so the band
sits below the point estimate. The point estimate is left uncorrected rather
than bias-adjusted, since a two-transition bias estimate is not a stable enough
basis to shift every forecast.

In [12]:
out = pd.read_csv("../data/processed/forecast_tcas70.csv")
print(f"{len(out)} rows · {out.code.nunique()} codes · all intervals present: "
      f"{out[['lo80','hi80','lo95','hi95']].notna().all().all()}")
out.tail(4).round(2)

48 rows · 48 codes · all intervals present: True


,code,program,faculty,point,lo80,hi80,lo95,hi95,model,beat_baseline
44,10010137111901A,หลักสูตรวิทยาศาสตรบัณฑิต สาขาวิชารังสีเทคนิค,คณะสหเวชศาสตร์,52.82,42.04,58.83,33.53,60.79,lvcf,False
45,10010138113101A,หลักสูตรวิทยาศาสตรบัณฑิต สาขาวิชาจิตวิทยา,คณะจิตวิทยา,57.93,47.16,63.94,38.65,65.90,lvcf,False
46,10010139112001A,หลักสูตรวิทยาศาสตรบัณฑิต สาขาวิชาวิทยาศาสตร์กา...,คณะวิทยาศาสตร์การกีฬา,58.97,48.20,64.99,39.69,66.94,lvcf,False
47,10010140900301A,หลักสูตรศิลปศาสตรและวิทยาศาสตรบัณฑิต สาขาวิชาก...,สำนักวิชาทรัพยากรการเกษตร,53.40,42.62,59.41,34.11,61.37,lvcf,False


## What this notebook concludes

1. **The hierarchical model does not beat the baselines** — 6.66 MAE against
   4.89. `beat_baseline` is `False` in the output CSV.
2. **Trend estimation is the problem, not the implementation.** Ranking methods
   by how freely they fit a per-program slope reproduces the error ranking
   exactly. Partial pooling helps (it beats unpooled OLS at both origins); it
   just cannot rescue a slope fitted on three or four points.
3. **`lvcf` ships**, chosen on two origins rather than one, because the single
   spec holdout picks a winner that is statistically tied and less stable.
4. **The interval is wide and stays wide.** 16.8 points at 80%.

### The highest-value next step is not a better model

It is more data. `assets.mytcas.com/maxmin/TCAS{64,65}_maxmin.xlsx` would take
the panel from 4 points to 6, which is the difference between a slope that
cannot be estimated and one that might be. TCAS62–63 predate TGAT/TPAT
entirely, so their composite is not comparable and they do not help.

No deep net was tried, per the spec — and on 4 points per program that
constraint needs no defending. If one is added later it must be benchmarked
against these same baselines on these same splits.